In [1]:
import os
import re
import time
import html
import requests
import json
import urllib.parse
import urllib.request
from datetime import datetime
import trafilatura

In [2]:
client_id = "xNPv1i9IOBPESq3aW08Q"
client_secret ="CO6stAZN55"


QUERIES = ["야구", "KBO 야구", "키움"]
DISPLAY = 100
SORT = "date"  # 최신순

def search_news(query):
    enc = urllib.parse.quote(query)
    url = (
        f"https://openapi.naver.com/v1/search/news.json"
        f"?query={enc}&display={DISPLAY}&start=1&sort={SORT}"
    )

    req = urllib.request.Request(url)
    req.add_header("X-Naver-Client-Id", client_id)
    req.add_header("X-Naver-Client-Secret", client_secret)

    with urllib.request.urlopen(req) as res:
        return json.loads(res.read().decode("utf-8"))

all_items = []
for q in QUERIES:
    data = search_news(q)
    items = data.get("items", [])
    print(f"쿼리 '{q}' 결과 수:", len(items))
    all_items.extend(items)

print("\n합치기 전 전체 개수:", len(all_items))

# -----------------------------
# 1️⃣ URL 기준 중복 제거
# -----------------------------
seen = set()
deduped = []
for it in all_items:
    url = it.get("originallink") or it.get("link")
    if not url:
        continue
    if url in seen:
        continue
    seen.add(url)
    deduped.append(it)

print("중복 제거 후 개수:", len(deduped))

BASEBALL_KEYWORDS = [
    "야구", "kbo", "프로야구", "투수", "타자",
    "홈런", "선발", "불펜", "경기", "시즌",
    "키움", "히어로즈", "lg", "두산", "ssg", "kt"
]

def is_baseball_article(it):
    text = ((it.get("title") or "") + " " + (it.get("description") or "")).lower()
    return any(k in text for k in BASEBALL_KEYWORDS)

baseball_only = [it for it in deduped if is_baseball_article(it)]
print("야구 기사 필터 후:", len(baseball_only))


print("\n샘플 5개 ↓")
for it in baseball_only[:5]:
    print("- title:", it.get("title"))
    print("  link:", it.get("link"))
    print("  originallink:", it.get("originallink"))


쿼리 '야구' 결과 수: 100
쿼리 'KBO 야구' 결과 수: 100
쿼리 '키움' 결과 수: 100

합치기 전 전체 개수: 300
중복 제거 후 개수: 262
야구 기사 필터 후: 262

샘플 5개 ↓
- title: 충남도 '천안아산역 돔구장 건립' 전문가 자문회의
  link: https://news.skbroadband.com/news/articleView.html?idxno=214086
  originallink: https://news.skbroadband.com/news/articleView.html?idxno=214086
- title: &quot;류현진이 울컥했다&quot;…한화의 암흑과 비상을 기록한 책, 《이글스라 행...
  link: https://www.dailysportshankook.co.kr/news/articleView.html?idxno=421399
  originallink: https://www.dailysportshankook.co.kr/news/articleView.html?idxno=421399
- title: 2026 프로<b>야구</b> 외국인 선수 구성 40명 전원 확정
  link: http://www.m-i.kr/news/articleView.html?idxno=1320190
  originallink: http://www.m-i.kr/news/articleView.html?idxno=1320190
- title: &quot;좋은 말만은 안 된다&quot; 추신수 감독, '블랙퀸즈' 지옥훈련
  link: https://www.gpkorea.com/news/articleView.html?idxno=138178
  originallink: https://www.gpkorea.com/news/articleView.html?idxno=138178
- title: [매경주최 세계 기선전] AI 활용·20초·빠른 승부…바둑판 바꾼 세계 기...
  link: https://m.sports.naver.com/gen

In [3]:
import requests
import trafilatura

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; BaseballNewsBot/0.1)"
}

def fetch_article_text(url: str) -> str | None:
    try:
        r = requests.get(url, headers=HEADERS, timeout=10)
        if r.status_code != 200:
            return None

        html = r.text
        text = trafilatura.extract(
            html,
            include_comments=False,
            include_tables=False
        )

        if not text:
            return None

        text = text.strip()

        # 너무 짧으면 본문 실패로 간주
        if len(text) < 300:
            return None

        return text

    except requests.RequestException:
        return None

In [4]:
print("\n 본문 크롤링 테스트 ↓")

for it in deduped[:5]:
    title = it.get("title")
    originallink = it.get("originallink")

    print("\n" + "="*80)
    print("제목:", title)
    print("원문 URL:", originallink)

    if not originallink:
        print("originallink 없음")
        continue

    text = fetch_article_text(originallink)

    if text:
        print(f"본문 추출 성공 (len={len(text)})")
        print("본문 앞 300자:")
        print(text[:300].replace("\n", " "))
    else:
        print("본문 추출 실패 (차단 / 구조 / 동적 렌더링)")



 본문 크롤링 테스트 ↓

제목: 충남도 '천안아산역 돔구장 건립' 전문가 자문회의
원문 URL: https://news.skbroadband.com/news/articleView.html?idxno=214086
본문 추출 성공 (len=405)
본문 앞 300자:
[기사 내용] 충청남도가 천안아산역 인근에 프로야구와 공연, 전시가 가능한 다목적 돔구장 건립을 위해 첫 전문가 자문회의를 열었습니다. 도는 내년 타당성 조사와 기본계획 수립에 이어 2027년 예비타당성 조사를 거쳐 2031년 준공을 목표로 하고 있습니다. 365일 기상 여건의 영향을 받지 않는 '대한민국 복합 여가 플랫폼'을 구축한다는 구상입니다. 김태흠 지사는 “K-팝 공연을 국내에서 소화할 대형 인프라가 부족하다”며 “국력과 경제력에 걸맞은 돔 시설이 필요하다”고 강조했습니다. 앞서 문화체육관광부도 업무계획 보고를 통해 "높은

제목: &quot;류현진이 울컥했다&quot;…한화의 암흑과 비상을 기록한 책, 《이글스라 행...
원문 URL: https://www.dailysportshankook.co.kr/news/articleView.html?idxno=421399
본문 추출 성공 (len=1224)
본문 앞 300자:
[데일리스포츠한국 심응섭 기자] 류현진이 “한화 이글스 그 자체가 담긴 책”이라며 직접 추천한 한화 이글스 역사서가 세상에 나왔다. 미담도, 실패담도 아닌 20년의 시간을 정면으로 기록한 책 《이글스라 행복합니다》(북오션)가 출간되며 팬들의 시선을 끌고 있다. 이 책은 중앙일보에서 20년간 프로야구를 취재해 온 배영은 기자와 한화 이글스의 레전드 투수이자 전 단장인 정민철 해설위원이 공동 집필했다. 류현진이 데뷔한 2006년부터 올해까지, 한화 이글스의 20시즌이 한 권에 오롯이 담겼다. 저자들은 화려한 성공담이나 감정적인 미화 대

제목: 2026 프로<b>야구</b> 외국인 선수 구성 40명 전원 확정
원문 URL: http://www.m-i.kr/news/article